# VoiceSecure 훈련 노트북 (SDK Evaluator 파이프라인)

**실행 순서대로 셀을 실행하세요.**

### 사전 준비
- 런타임 유형 = **GPU (T4 이상)**
- 다음 중 하나로 데이터 준비:
  - **AIHub 014 다화자 음성합성 데이터** — Step 2-B의 aihubshell로 직접 다운로드
  - **KSS 데이터셋** — Google Drive에 미리 업로드

### 파이프라인 (의도된 매핑)
```
원본 음성
  └─▶ RLAgent.act()             — state(36-dim) → 노이즈 action(257×100)
  └─▶ PsychoacousticMasker      — 심리음향 임계치로 clamp
  └─▶ Mixer                     — STFT 도메인 합성 → 변조 음성
       │
       ├─ SpeakerEvaluator       (WavLM-SV + CAM++)      → sv_score
       ├─ TTSEvaluator           (XTTS, 10 ep마다)        → tts_score
       └─ ASREvaluator           (wav2vec2-xlsr-korean)   → asr_cer
            └▶ RewardFunction (alpha·sv + beta·tts - lambda·max(0, cer-tau))
                  └▶ PPO update
```

기존 CosyVoice + ECAPA 직접 호출 흐름은 제거됨. 모두 SDK Evaluator 통과.


## Step 1. GPU 확인

In [ ]:
import torch, sys

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: GPU 없음 — 런타임 > 런타임 유형 변경 > GPU 설정 필요')


## Step 2. Google Drive 마운트 + 데이터 경로 설정

아래 셀 실행 후 Drive 연결 허용을 눌러주세요.

**DATA_FORMAT** (kss / aihub) 과 **DATA_DIR**을 본인 데이터에 맞게 수정하세요.

> Drive에 안 올리고 Colab에 직접 받고 싶으면 → **Step 2-B (aihubshell)** 이용


In [ ]:
# ── Zeroth-Korean 받기 + train.py 호환 폴더로 변환 ─────────────────
import os, shutil
from pathlib import Path
from collections import defaultdict

ZEROTH_ROOT = Path('/content/zeroth')
ZEROTH_ROOT.mkdir(exist_ok=True)
%cd {ZEROTH_ROOT}

# 1. 다운로드 (12GB, 10분)
if not Path('zeroth_korean.tar.gz').exists() and not list(ZEROTH_ROOT.rglob('*.wav'))[:1]:
    !wget -q --show-progress https://www.openslr.org/resources/40/zeroth_korean.tar.gz

# 2. 압축 해제 (5분)
if not list(ZEROTH_ROOT.rglob('*.wav'))[:1]:
    !tar xzf zeroth_korean.tar.gz
    !rm zeroth_korean.tar.gz

print('wav 개수:', len(list(ZEROTH_ROOT.rglob('*.wav'))))

# 3. KSS 형식 폴더로 평탄화 (화자 30명 × 100파일)
DST = Path('/content/zeroth_kss_format')
if DST.exists(): shutil.rmtree(DST)
DST.mkdir()

# Labels.txt
labels = []
for trans in ZEROTH_ROOT.rglob('*.trans.txt'):
    for line in trans.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            labels.append(f'{parts[0]} {parts[1]}')
(DST / 'Labels.txt').write_text('\n'.join(labels), encoding='utf-8')

# 화자 30명 × 100파일
by_spk = defaultdict(list)
for p in sorted(ZEROTH_ROOT.rglob('*.wav')):
    by_spk[p.stem.split('-')[0]].append(p)

for spk in sorted(by_spk)[:30]:
    dst_spk = DST / spk
    dst_spk.mkdir()
    for src in by_spk[spk][:100]:
        shutil.copy(src, dst_spk / src.name)

print(f'화자 {len(list(DST.iterdir()))-1}명, '
      f'wav {sum(1 for _ in DST.rglob("*.wav"))} 파일')
print(f'크기: {sum(p.stat().st_size for p in DST.rglob("*"))/1e6:.1f} MB')

# 4. Step 2 변수 설정
DATA_FORMAT = 'kss'                      # Zeroth는 KSS 형식으로 변환
DATA_DIR    = str(DST)                   # /content/zeroth_kss_format
MAX_SPEAKERS          = None
MAX_FILES_PER_SPEAKER = None
print(f'\nDATA_DIR = {DATA_DIR}')

In [ ]:
import shutil
from pathlib import Path
from collections import defaultdict

ZEROTH_ROOT = Path('/content/zeroth')
DST = Path('/content/zeroth_kss_format')
if DST.exists():
    shutil.rmtree(DST)
DST.mkdir()

# Labels.txt (이미 만들어졌지만 재생성)
labels = []
for trans in ZEROTH_ROOT.rglob('*.trans.txt'):
    for line in trans.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            labels.append(f'{parts[0]} {parts[1]}')
(DST / 'Labels.txt').write_text('\n'.join(labels), encoding='utf-8')
print(f'[labels] {len(labels)} 줄')

# ★ flac 우선, 없으면 wav
audio_files = sorted(ZEROTH_ROOT.rglob('*.flac'))
ext = '.flac'
if len(audio_files) == 0:
    audio_files = sorted(ZEROTH_ROOT.rglob('*.wav'))
    ext = '.wav'
print(f'[audio] {len(audio_files)}개 ({ext})')

# 화자 30명 × 100파일
by_spk = defaultdict(list)
for p in audio_files:
    by_spk[p.stem.split('-')[0]].append(p)
print(f'[speakers] 전체 {len(by_spk)}명, 처음 5명 wav 수:',
      [len(by_spk[s]) for s in sorted(by_spk)[:5]])

for spk in sorted(by_spk)[:30]:
    dst_spk = DST / spk
    dst_spk.mkdir()
    for src in by_spk[spk][:100]:
        # ★ flac이면 wav로 변환해서 저장 (train.py가 *.wav만 glob)
        if ext == '.flac':
            import soundfile as sf
            data, sr = sf.read(str(src))
            sf.write(str(dst_spk / (src.stem + '.wav')), data, sr)
        else:
            shutil.copy(src, dst_spk / src.name)

n_wavs = sum(1 for _ in DST.rglob('*.wav'))
n_speakers = len([d for d in DST.iterdir() if d.is_dir()])
print(f'[done] 화자 {n_speakers}명, wav {n_wavs}개')
print(f'[size] {sum(p.stat().st_size for p in DST.rglob("*"))/1e6:.1f} MB')

In [ ]:
# 어떤 파일이 화자 폴더 안에 들어있는지 확인
!ls /content/zeroth/train_data_01/003/142/ | head -10

# 전체에서 wav, flac, 그 외 파일 형식 카운트
!find /content/zeroth -type f -name "*.wav"  | wc -l
!find /content/zeroth -type f -name "*.flac" | wc -l
!find /content/zeroth -type f | head -20

In [ ]:
!ls /content/zeroth_kss_format | head
!ls /content/zeroth_kss_format/ | head -1 | xargs -I {} ls /content/zeroth_kss_format/{} | head -3

In [ ]:
import shutil
from pathlib import Path
from collections import defaultdict
import soundfile as sf

ZEROTH_ROOT = Path('/content/zeroth')
DST = Path('/content/zeroth_kss_format')
if DST.exists():
    shutil.rmtree(DST)
DST.mkdir()

# Labels.txt
labels = []
for trans in ZEROTH_ROOT.rglob('*.trans.txt'):
    for line in trans.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            labels.append(f'{parts[0]} {parts[1]}')
(DST / 'Labels.txt').write_text('\n'.join(labels), encoding='utf-8')
print(f'[labels] {len(labels)} 줄')

audio_files = sorted(ZEROTH_ROOT.rglob('*.flac'))
print(f'[audio] {len(audio_files)}개')

# ★ 화자 ID = 파일명 첫 '_' 토큰 (142_003_0003 → 142)
by_spk = defaultdict(list)
for p in audio_files:
    by_spk[p.stem.split('_')[0]].append(p)

speakers = sorted(by_spk.keys())
print(f'[speakers] 전체 {len(speakers)}명')
print(f'  처음 10명 파일 수: {[(s, len(by_spk[s])) for s in speakers[:10]]}')

# 화자 30명 × 100파일, flac → wav 변환
for spk in speakers[:30]:
    dst_spk = DST / spk
    dst_spk.mkdir()
    for src in by_spk[spk][:100]:
        data, sr = sf.read(str(src))
        sf.write(str(dst_spk / (src.stem + '.wav')), data, sr)

n_wavs = sum(1 for _ in DST.rglob('*.wav'))
n_speakers = len([d for d in DST.iterdir() if d.is_dir()])
print(f'\n[done] 화자 {n_speakers}명, wav {n_wavs}개')
print(f'[size] {sum(p.stat().st_size for p in DST.rglob("*"))/1e6:.1f} MB')

In [ ]:
# 화자별 파일 수 분포 (1개씩이 아니라 수백개씩이어야 정상)
import collections
counts = sorted(collections.Counter(p.stem.split('_')[0]
                                     for p in audio_files).values(), reverse=True)
print(f'화자 수: {len(counts)}')
print(f'화자당 파일 수 상위 5: {counts[:5]}')
print(f'화자당 파일 수 하위 5: {counts[-5:]}')

## Step 3. VoiceSecure SDK 설치

GitHub에서 SDK를 clone하고 editable 모드로 설치합니다.
이미 있으면 `git pull`로 최신화.


In [ ]:
import os

SDK_ROOT = '/content/voicesecure-sdk'

if not os.path.exists(SDK_ROOT):
    !git clone https://github.com/VoiceSecureHoseo/voicesecure-sdk.git {SDK_ROOT}
else:
    !git -C {SDK_ROOT} pull
    print('SDK 업데이트 완료')

%cd {SDK_ROOT}
!pip install -q -e .
print('SDK 설치 완료')


## Step 4. 추가 의존성 설치

- **coqui-tts** — XTTS v2 (TTSEvaluator 백엔드). XTTS는 CPML 라이선스이므로 `COQUI_TOS_AGREED=1` 환경변수로 동의 표시.
- `transformers`, `scipy`, `soundfile`, `onnxruntime`, `tensorboard` 는 SDK pyproject 의존성이라 Step 3에서 이미 설치됨.

CosyVoice/SpeechBrain/ECAPA 의존성은 새 파이프라인에서 **사용 안 함**.


In [ ]:
!pip install -q coqui-tts

import os
os.environ['COQUI_TOS_AGREED'] = '1'
print('[OK] coqui-tts installed, XTTS TOS agreed')


## Step 5. 어댑터 로드 검증

4개 어댑터를 순서대로 로드합니다.
첫 실행 시 모델 가중치 다운로드:
- WavLM-SV: 380 MB (~1분)
- CAM++ ONNX: 30 MB (~30초)
- XTTS v2: 1.8 GB (~3분)
- wav2vec2-xlsr-korean: 1.2 GB (~2분)

총 약 7-10분.


In [ ]:
import sys, os
sys.path.insert(0, '/content/voicesecure-sdk/src')

# 확인 (둘 다 True여야 함)
print('SDK 폴더 존재:', os.path.exists('/content/voicesecure-sdk/src/voicesecure'))
import voicesecure
print('voicesecure 모듈 경로:', voicesecure.__file__)
print('train.py 존재:', os.path.exists('/content/voicesecure-sdk/train.py'))

In [ ]:
import time

from voicesecure.evaluators.adapters.wavlm_sv import WavLMSVAdapter
from voicesecure.evaluators.adapters.campplus import CAMPlusAdapter
from voicesecure.evaluators.adapters.xtts import XTTSAdapter
from voicesecure.evaluators.adapters.wav2vec2_asr import Wav2Vec2KoreanAdapter

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

t0 = time.time(); wavlm = WavLMSVAdapter(device=device);                   print(f'[load] WavLM-SV    {time.time()-t0:5.1f}s')
t0 = time.time(); cam   = CAMPlusAdapter(device=device);                   print(f'[load] CAM++        {time.time()-t0:5.1f}s')
t0 = time.time(); xtts  = XTTSAdapter();                                    print(f'[load] XTTS v2      {time.time()-t0:5.1f}s')
t0 = time.time(); asr   = Wav2Vec2KoreanAdapter(device=device);             print(f'[load] wav2vec2-ko  {time.time()-t0:5.1f}s')
print('어댑터 4종 로드 완료')


## Step 6. 빠른 동작 확인 (1 에피소드)

훈련 전 SDK Evaluator 체인이 정상 동작하는지 확인.
합성 오디오(1초)로 act → clamp → mix → SpeakerEvaluator → ASREvaluator → TTSEvaluator → RewardFunction.


In [ ]:
import numpy as np

from voicesecure.evaluators import SpeakerEvaluator, TTSEvaluator, ASREvaluator
from voicesecure.modulation.masker import PsychoacousticMasker
from voicesecure.modulation.mixer import Mixer
from voicesecure.reward.function import RewardFunction
from voicesecure.rl.agent import RLAgent
from voicesecure.types import SAMPLE_RATE
from train import emb_dist_normalizer    # cosine-distance → [0,1] 스케일 함수

# 합성 테스트 오디오 (1초)
rng = np.random.default_rng(42)
t = np.arange(SAMPLE_RATE, dtype=np.float32) / SAMPLE_RATE
test_audio = (
    0.3 * np.sin(2 * np.pi * 200 * t)
    + 0.2 * np.sin(2 * np.pi * 800 * t)
    + 0.02 * rng.standard_normal(SAMPLE_RATE).astype(np.float32)
).astype(np.float32)
test_text = '안녕하세요'

# Evaluator 조합 (의도된 매핑)
sv_eval  = SpeakerEvaluator(wavlm_model=wavlm, cam_model=cam, normalizer=emb_dist_normalizer)
tts_eval = TTSEvaluator(xtts_model=xtts, speaker_model=wavlm,
                        normalizer=emb_dist_normalizer, sampling_interval=10)
asr_eval = ASREvaluator(asr_model=asr)
reward_fn = RewardFunction()   # SDK default: alpha=0.6, beta=0.4, lambda_asr=1.0, cer_threshold=0.3

# 모듈
agent  = RLAgent()
masker = PsychoacousticMasker()
mixer  = Mixer()

# 1 에피소드
state, action, log_prob, value, freq_pattern, time_gate = agent.act(test_audio)
safe_noise = masker.clamp(test_audio, action)
modified   = mixer.mix(test_audio, safe_noise)

sv_out  = sv_eval.evaluate(test_audio, modified)
tts_out = tts_eval.evaluate(test_audio, modified)
asr_out = asr_eval.evaluate(test_audio, modified, original_text=test_text)

components = {
    'sv_score':  float(np.clip(sv_out.score, 0, 1)),
    'tts_score': float(np.clip(tts_out.score, 0, 1)),
    'asr_cer':   float(np.clip(asr_out.raw_metric, 0, 1)),
}
reward = reward_fn.compute(components)

print(f'action shape : {tuple(action.shape)}  (기대: (257, 100))')
print(f'sv_score     : {sv_out.score:.4f}  (raw={sv_out.raw_metric:.4f})')
print(f'tts_score    : {tts_out.score:.4f}  (raw={tts_out.raw_metric:.4f})')
print(f'asr_cer      : {asr_out.raw_metric:.4f}  (score={asr_out.score:.4f})')
print(f'reward       : {reward:.4f}')
print('파이프라인 정상 동작 확인')


In [ ]:
import os
from pathlib import Path

# 1) DATA_DIR이 뭘 가리키는지 + 안에 뭐가 있는지
print('DATA_DIR     :', DATA_DIR)
print('exists       :', os.path.exists(DATA_DIR))
if os.path.exists(DATA_DIR):
    items = os.listdir(DATA_DIR)
    print(f'안 항목 수   : {len(items)}')
    print(f'처음 10개    : {items[:10]}')
    print(f'Labels.txt   : {os.path.exists(f"{DATA_DIR}/Labels.txt")}')
    if os.path.exists(f'{DATA_DIR}/Labels.txt'):
        sz = os.path.getsize(f'{DATA_DIR}/Labels.txt')
        print(f'Labels.txt 크기 : {sz} bytes')
        !head -3 "{DATA_DIR}/Labels.txt"
    # 화자 폴더별 wav 수
    for item in items[:5]:
        p = f'{DATA_DIR}/{item}'
        if os.path.isdir(p):
            wavs = list(Path(p).glob('*.wav'))
            print(f'  {item}/  wav {len(wavs)}개')

print()
# 2) Zeroth 원본 다운로드 폴더 상태
print('--- Zeroth 원본 ---')
print('/content/zeroth 존재:', os.path.exists('/content/zeroth'))
if os.path.exists('/content/zeroth'):
    !find /content/zeroth -maxdepth 3 -type d | head -10
    print()
    n_wav = len(list(Path('/content/zeroth').rglob('*.wav')))
    print(f'전체 wav 개수: {n_wav}')

## Step 7. 훈련 실행

### 새 train.py CLI 인자
| 인자 | 기본값 | 설명 |
|---|---|---|
| `--data_format` | `kss` | kss / aihub |
| `--max_speakers` | None | (aihub) 처음 N명만 |
| `--max_files_per_speaker` | None | (aihub) 화자당 N개 |
| `--epochs` | 3 | 1 에폭 = 데이터셋 전체 |
| `--checkpoint_interval` | 1000 | 에피소드 단위 |
| `--lr` | 3e-4 | PPO Adam lr |
| `--resume` | None | 이어서 훈련 |
| `--xtts_model_dir` | None (자동 탐색) | XTTS 모델 경로 |
| `--use_tts / --no-use-tts` | True | TTS 평가 끄기 |
| `--use_asr / --no-use-asr` | True | ASR 평가 끄기 |
| `--tts_eval_interval` | 10 | TTS 평가 주기 |
| `--reward_alpha / --reward_beta / --reward_lambda_asr / --reward_cer_threshold` | 0.6 / 0.4 / 1.0 / 0.3 | reward 가중치 |

학습이 느리면 `--no-use-asr` 또는 `--tts_eval_interval 50` 으로 가볍게 시작 가능.


In [ ]:
import os, shutil

# 1) 옛 캐시 삭제 — 새로 빌드해야 Zeroth file_id로 채워짐
ckpt_cache = '/content/voicesecure-sdk/checkpoints/embedding_cache_sdk.pt'
if os.path.exists(ckpt_cache):
    os.remove(ckpt_cache)
    print('[clean] 옛 014 캐시 삭제')

# 2) Drive 백업 폴더에도 옛 캐시 있으면 같이 삭제 (안 그러면 다음 학습 때 또 복원됨)
backup_cache = '/content/drive/MyDrive/voicesecure_checkpoints/embedding_cache_sdk.pt'
if os.path.exists(backup_cache):
    os.remove(backup_cache)
    print('[clean] Drive 캐시 삭제')

# 3) 변환 결과 점검 — 화자별 wav 수
DST = '/content/zeroth_kss_format'
n = 0
short_speakers = []
for d in sorted(os.listdir(DST)):
    p = f'{DST}/{d}'
    if os.path.isdir(p):
        cnt = len([f for f in os.listdir(p) if f.endswith('.wav')])
        n += cnt
        if cnt < 100:
            short_speakers.append((d, cnt))
print(f'\n전체 wav: {n}개')
if short_speakers:
    print(f'100개 못 채운 화자: {len(short_speakers)}명')
    print(f'  처음 5: {short_speakers[:5]}')

# 4) Labels.txt 매칭 검증 — 몇 개가 매칭되는지
labels = {}
with open(f'{DST}/Labels.txt', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            labels[parts[0]] = parts[1]
print(f'Labels.txt: {len(labels)} 줄')

matched = 0
for d in os.listdir(DST):
    p = f'{DST}/{d}'
    if os.path.isdir(p):
        for f in os.listdir(p):
            if f.endswith('.wav') and f[:-4] in labels:
                matched += 1
print(f'wav ↔ Labels.txt 매칭: {matched}개 (학습에 쓰일 수)')

In [ ]:
import os, shutil
SDK_ROOT       = '/content/voicesecure-sdk'
CHECKPOINT_DIR = f'{SDK_ROOT}/checkpoints'
BACKUP_DIR     = '/content/drive/MyDrive/voicesecure_checkpoints'

# 이전 임베딩 캐시 있으면 복원 (Zeroth용은 새로 만들어야 함)
# 이 데이터셋과 매칭 안 되니 굳이 복원 안 함

!python {SDK_ROOT}/train.py \
    --data_dir       {DATA_DIR} \
    --data_format    kss \
    --epochs         3 \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --tts_eval_interval 10

# 학습 끝나면 즉시 Drive 백업
if os.path.exists(CHECKPOINT_DIR):
    shutil.copytree(CHECKPOINT_DIR, BACKUP_DIR, dirs_exist_ok=True)
    print(f'[backup] Drive 백업 완료: {BACKUP_DIR}')

### 이어서 훈련 (런타임 재연결 후)

Colab 런타임이 끊겼다 재연결 시 아래 셀로 재개.


In [ ]:
RESUME_CKPT = f'{CHECKPOINT_DIR}/best.pt'   # 또는 episode_N.pt

if DATA_FORMAT == 'aihub':
    extra_args = '--data_format aihub'
    if MAX_SPEAKERS is not None:
        extra_args += f' --max_speakers {MAX_SPEAKERS}'
    if MAX_FILES_PER_SPEAKER is not None:
        extra_args += f' --max_files_per_speaker {MAX_FILES_PER_SPEAKER}'
else:
    extra_args = '--data_format kss'

!python {SDK_ROOT}/train.py \
    --data_dir       {DATA_DIR} \
    --epochs         3 \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --resume         {RESUME_CKPT} \
    {extra_args}


## Step 9. TensorBoard 로그 확인

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {CHECKPOINT_DIR}/logs


## Step 10. 방어 효과 평가 (XTTS 클로닝)

훈련된 에이전트로 데이터셋 파일 하나를 변조한 뒤,
XTTS 클로닝 전/후 WavLM 화자 임베딩 거리를 비교.

**목표 지표**: cosine distance > 0.3 (SIM < 0.25 논문 기준)


In [ ]:
# ── 학습된 정책으로 변조 음성 + XTTS 클론 평가 + 청취 ────────────────
import os, sys
import numpy as np
import soundfile as sf
import torch
from math import gcd
from pathlib import Path
from scipy.signal import resample_poly
from IPython.display import Audio, display

sys.path.insert(0, '/content/voicesecure-sdk/src')

from voicesecure.rl.agent import RLAgent
from voicesecure.modulation.masker import PsychoacousticMasker
from voicesecure.modulation.mixer import Mixer
from voicesecure.evaluators.adapters.wavlm_sv import WavLMSVAdapter
from voicesecure.evaluators.adapters.xtts import XTTSAdapter
from voicesecure.evaluators.base import cosine_distance

SR = 16000

def load_audio(path, sr=SR):
    data, src_sr = sf.read(str(path), dtype='float32', always_2d=True)
    audio = data.mean(axis=1).astype(np.float32)
    if src_sr != sr:
        g = gcd(sr, src_sr)
        audio = resample_poly(audio, sr // g, src_sr // g).astype(np.float32)
    return np.clip(audio, -1.0, 1.0).astype(np.float32)

# ── 1) best.pt 자동 탐색: Drive → Colab → 마지막 episode_*.pt ──────
import glob
def find_ckpt():
    for p in ['/content/drive/MyDrive/voicesecure_checkpoints/best.pt',
              '/content/voicesecure-sdk/checkpoints/best.pt']:
        if os.path.exists(p):
            return p
    eps = sorted(glob.glob('/content/**/episode_*.pt', recursive=True),
                 key=lambda p: int(p.rsplit('_', 1)[1].split('.')[0]))
    return eps[-1] if eps else None

ckpt = find_ckpt()
assert ckpt, 'best.pt / episode_*.pt 없음 — 학습이 끝났는지 확인'
print(f'[ckpt] {ckpt}')

# ── 2) 메모리에 어댑터/모듈 살아있으면 재사용, 없으면 새로 로드 ────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if 'agent_eval' not in globals():
    agent_eval = RLAgent()
agent_eval.load(ckpt)

if 'wavlm'  not in globals(): wavlm  = WavLMSVAdapter(device=device)
if 'xtts'   not in globals(): xtts   = XTTSAdapter()
if 'masker' not in globals(): masker = PsychoacousticMasker()
if 'mixer'  not in globals(): mixer  = Mixer()
print('[ready] agent + 4종 어댑터')

# ── 3) KSS 형식: Labels.txt 에서 텍스트 가져오기 ────────────────────
DATA_DIR = globals().get('DATA_DIR', '/content/zeroth_kss_format')
labels = {}
with open(f'{DATA_DIR}/Labels.txt', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            labels[parts[0]] = parts[1]

# 화자 중 1명 골라 wav 1개 선택 (원하면 직접 경로 지정)
wavs = sorted(Path(DATA_DIR).rglob('*.wav'))
test_wav = wavs[0]
test_text = labels.get(test_wav.stem, '안녕하세요')
print(f'\n[file] {test_wav.name}')
print(f'[text] {test_text}')

# ── 4) 원본 → 변조 ─────────────────────────────────────────────────
original = load_audio(test_wav)
orig_emb = wavlm.extract_embedding(original)

_, action, _, _, _, _ = agent_eval.act(original, deterministic=True)
safe_noise = masker.clamp(original, action)
modified   = mixer.mix(original, safe_noise)

mod_emb     = wavlm.extract_embedding(modified)
direct_dist = cosine_distance(orig_emb, mod_emb)

# ── 5) 변조 음성 → XTTS clone (공격 시뮬레이션) ────────────────────
xtts.clone_text = test_text
cloned = xtts.clone(modified)
clone_emb = wavlm.extract_embedding(cloned)
clone_dist = cosine_distance(orig_emb, clone_emb)

# ── 6) 수치 + 진단 ────────────────────────────────────────────────
print()
print('━━━ 방어 효과 평가 ━━━━━━━━━━━━━━━━━━━━━━')
print(f'직접 거리 (원본 vs 변조)      : {direct_dist:.4f}  ← 낮을수록 변조 티 안 남 (귀로 들리지 않음)')
print(f'클론 거리 (원본 vs 변조→XTTS): {clone_dist:.4f}  ← 핵심 지표')
print()
print('해석:')
print('  > 0.30  → 다른 사람 (방어 성공, 논문 SIM<0.25 기준)')
print('  0.10~0.30 → 애매 (부분 방어)')
print('  < 0.10  → 같은 사람 (방어 실패)')
print(f'\n→ {"성공 ✓" if clone_dist > 0.30 else "미달 — 학습 더 필요 또는 EMB_DIST_SCALE/TTS_EVAL_INTERVAL 조정"}')

# action / noise 크기도 같이 (학습이 진짜 0이 아닌 노이즈를 만드는지)
print()
print(f'action.abs().max() = {float(action.abs().max()):.4f}  (학습됐으면 0.1 이상)')
print(f'safe_noise.abs().max() = {float(safe_noise.abs().max()):.6f}')

# ── 7) 청취 위젯 3개 ───────────────────────────────────────────────
print()
print('═══════ 1) 원본 ═══════')
display(Audio(original, rate=SR))
print('═══════ 2) 변조 (귀로는 원본과 거의 동일해야 함) ═══════')
display(Audio(modified, rate=SR))
print('═══════ 3) XTTS가 변조 음성으로 클로닝한 결과 ═══════')
display(Audio(cloned, rate=SR))

# ── 8) Drive에 wav로 저장 ─────────────────────────────────────────
out_dir = '/content/drive/MyDrive/voicesecure_audio_samples'
os.makedirs(out_dir, exist_ok=True)
stem = test_wav.stem
sf.write(f'{out_dir}/{stem}_original.wav',    original, SR)
sf.write(f'{out_dir}/{stem}_modified.wav',    modified, SR)
sf.write(f'{out_dir}/{stem}_xtts_clone.wav',  cloned,   SR)
print(f'\n[saved] {out_dir}/{stem}_*.wav (3 파일)')

## Step 11. 체크포인트 Drive 백업

`/content` 안의 파일은 런타임 종료 시 삭제됩니다.
훈련 후 아래 셀로 Drive에 백업.


In [ ]:
import shutil

BACKUP_DIR = '/content/drive/MyDrive/voicesecure_checkpoints'

shutil.copytree(CHECKPOINT_DIR, BACKUP_DIR, dirs_exist_ok=True)
print(f'백업 완료: {BACKUP_DIR}')
